data -> preprocessing -> model -> website -> deploy

In [1]:
import numpy as np
import pandas as pd

In [2]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')


we just join these two data frame for ease of access


In [3]:
movies = movies.merge(credits, on='title')

colm which are  selected
#genres
#id
#keywords
#title
#overview
#cast
#crew

In [4]:
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]


In [5]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [6]:
movies.dropna(inplace=True)

In [7]:
movies.duplicated().sum()

np.int64(0)

creating helper function for standaruze data frame colms

In [8]:
import ast
ast.literal_eval(movies['genres'][0])

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [9]:
import ast

def convert(obj):
    if isinstance(obj, str):
        obj = ast.literal_eval(obj)
    
    L = []
    for i in obj:
        if isinstance(i, dict):
            L.append(i['name'])
        else:
            L.append(i)
    return L

In [10]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)


In [11]:
def convert3(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
         L.append(i['name'])
         counter += 1
        else:
            break
    return L

In [12]:
movies['cast'] = movies['cast'].apply(convert3)

In [13]:
def fetch_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

In [14]:
movies['crew'] = movies['crew'].apply(fetch_director)
movies['overview'] = movies['overview'].apply(lambda x:x.split())  

In [15]:
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x])
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x])
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x])
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x])

In [16]:
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [17]:
new_df = movies[['movie_id', 'title', 'tags']]
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11636\381017236.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))


In [18]:
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11636\3214958533.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


NOW WORK ON SIMILAR TAG FINDING USING VECTORIZATION METHOD 

In [19]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english')


In [21]:
vectors = cv.fit_transform(new_df['tags']).toarray()

to avoid the similar word like action and actions we use steming technique

In [23]:
!pip install nltk

     ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
      --------------------------------------- 0.0/1.6 MB 640.0 kB/s eta 0:00:03
     - -------------------------------------- 0.0/1.6 MB 388.9 kB/s eta 0:00:04
     -- ------------------------------------- 0.1/1.6 MB 744.7 kB/s eta 0:00:02
     --- ------------------------------------ 0.1/1.6 MB 774.0 kB/s eta 0:00:02
     ----- ---------------------------------- 0.2/1.6 MB 1.1 MB/s eta 0:00:02
     ----------- ---------------------------- 0.5/1.6 MB 1.7 MB/s eta 0:00:01
     --------------- ------------------------ 0.6/1.6 MB 1.9 MB/s eta 0:00:01
     ------------------------- -------------- 1.0/1.6 MB 2.8 MB/s eta 0:00:01
     ---------------------------------------  1.5/1.6 MB 3.6 MB/s eta 0:00:01
     ---------------------------------------  1.5/1.6 MB 3.6 MB/s eta 0:00:01
     ---------------------------------------- 1.6/1.6 MB 3.1 MB/s eta 0:00:00
     ---------------------------------------- 0.0/278.4 kB ? et


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
!python.exe -m pip install --upgrade pip

     ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
      --------------------------------------- 0.0/1.8 MB 640.0 kB/s eta 0:00:03
     -- ------------------------------------- 0.1/1.8 MB 871.5 kB/s eta 0:00:02
     --- ------------------------------------ 0.2/1.8 MB 1.2 MB/s eta 0:00:02
     ------ --------------------------------- 0.3/1.8 MB 1.6 MB/s eta 0:00:01
     ------------ --------------------------- 0.6/1.8 MB 2.2 MB/s eta 0:00:01
     ---------------------- ----------------- 1.0/1.8 MB 3.7 MB/s eta 0:00:01
     ---------------------------------------  1.8/1.8 MB 5.7 MB/s eta 0:00:01
     ---------------------------------------  1.8/1.8 MB 5.7 MB/s eta 0:00:01
     ---------------------------------------- 1.8/1.8 MB 4.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 23.0.1
    Uninstalling pip-23.0.1:
      Successfully uninstalled pip-23.0.1


In [26]:
import nltk

In [27]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

In [28]:
new_df['tags'] = new_df['tags'].apply(stem)       

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_11636\2723271394.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


we use cosine similarity for findind distance in vector for finding similarity in tags

In [29]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)


In [30]:
def recommender(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [31]:
recommender('Batman Begins')

The Dark Knight
The Dark Knight Rises
Batman
Batman
Batman & Robin


In [32]:
import pickle

In [33]:
pickle.dump(new_df , open('movies.pkl','wb'))

In [34]:
pickle.dump(similarity , open('similarity.pkl','wb'))
